# pix2pixHD Training Guide

This notebook is **not fully executable code** — it is a step-by-step checklist for training
NVIDIA's pix2pixHD on the prepared BBBC039 dataset (**mask → fluorescence**).

All commands below are meant to be run in a terminal (inside this folder).

## Step 1 — Clone the original repository

In [5]:
!git clone https://github.com/NVIDIA/pix2pixHD.git
# inside Unit_5_pix2pixHD

Cloning into 'pix2pixHD'...
remote: Enumerating objects: 343, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 343 (delta 0), reused 0 (delta 0), pack-reused 340 (from 1)
Receiving objects: 100% (343/343), 55.68 MiB | 30.30 MiB/s, done.
Resolving deltas: 100% (156/156), done.


## Step 2 — Inspect data layout

After running `00_PrepData_BBBC039.ipynb` you should have:
```
datasets/bbbc039/
├── train_A/   # 160 binary masks (RGB, 512×512)
├── train_B/   # 160 fluorescence images (RGB, 512×512)
├── test_A/    # 40 masks
└── test_B/    # 40 fluorescence images
```

Because we use `--label_nc 0`, pix2pixHD treats the input as a raw image, not a semantic map.

## Step 3 — Python Patches

The original NVIDIA repo was written for Python 3.5 / PyTorch 0.4.
To make it run on Python 3.11+ and modern PyTorch we apply a few small patches via
`apply_patches.py` (sitting next to `pix2pixHD/`). The script is **idempotent** — running
it again after patches are applied is a no-op.

**Run this once after cloning.**

In [6]:
!python apply_patches.py

  patch pix2pixHD/train.py
  patch pix2pixHD/models/networks.py
  patch pix2pixHD/models/pix2pixHD_model.py
All patches applied.


## Step 4 — Training command

Key flags:
* `--label_nc 0` — input is RGB, not a semantic label map
* `--no_instance` — we have no instance maps
* `--loadSize 512 --fineSize 512` — resize and crop to 512x512
* `--batchSize 2` — adjust to your GPU memory (1-4 for 512x512)
* `--niter 40 --niter_decay 0` — 40 epochs at full LR (no decay)
* `--save_epoch_freq 100` — disable default periodic saves so only milestones hit
* `--gpu_ids 0` — use GPU 0

**Milestone checkpoints:** the patched `train.py` saves models at epochs **5, 10, 20, 40** automatically.

In [4]:
!cd pix2pixHD && python train.py \
    --name bbbc039_512 \
    --dataroot ../datasets/bbbc039 \
    --label_nc 0 \
    --no_instance \
    --loadSize 512 \
    --fineSize 512 \
    --batchSize 2 \
    --niter 40 \
    --niter_decay 0 \
    --save_epoch_freq 100 \
    --gpu_ids 0 \
    --checkpoints_dir ./checkpoints

Traceback (most recent call last):
  File "/home/azureuser/Projects/GAI4_course/Unit_5_pix2pixHD/pix2pixHD/train.py", line 15, in <module>
    from util.visualizer import Visualizer
  File "/home/azureuser/Projects/GAI4_course/Unit_5_pix2pixHD/pix2pixHD/util/visualizer.py", line 6, in <module>
    from . import html
  File "/home/azureuser/Projects/GAI4_course/Unit_5_pix2pixHD/pix2pixHD/util/html.py", line 1, in <module>
    import dominate
ModuleNotFoundError: No module named 'dominate'


### What to expect during training

* Checkpoints are saved in `pix2pixHD/checkpoints/bbbc039_512/`.
* The discriminator loss (`D_fake`, `D_real`) and generator loss (`G_GAN`, `G_VGG`, `G_FM`) are printed.
* pix2pixHD uses a **multi-scale discriminator** + **feature-matching loss** + **VGG perceptual loss** —
  much richer than the vanilla BCE loss from the MNIST GAN.
* Training 40 epochs on 160 images at 512×512 takes ~1–2 hours on a GPU.
* **GPU memory:** ~6 GB at batch size 2 (512×512).

## Step 5 — Generate test outputs

Run inference on the test set to produce images for evaluation.
You can specify `--which_epoch latest` or any saved milestone (e.g. `5`, `10`, `20`, `40`).

In [ ]:
!cd pix2pixHD && python test.py \
    --name bbbc039_512 \
    --dataroot ../datasets/bbbc039 \
    --label_nc 0 \
    --no_instance \
    --loadSize 512 \
    --fineSize 512 \
    --which_epoch latest \
    --how_many 40 \
    --checkpoints_dir ./checkpoints \
    --results_dir ./results

### Output layout

Results land in `pix2pixHD/results/bbbc039_512/test_latest/images/`.
For each test sample you get three files:
* `XXXX_input_label.png` — the mask (input A)
* `XXXX_synthesized_image.png` — the generated fluorescence
* `XXXX_real_image.png` — the ground-truth fluorescence

We do the evaluation in `02_Evaluate_pix2pixHD.ipynb`.

## Takeaway

pix2pixHD is a **conditional GAN** for high-resolution image-to-image translation.
Unlike the MNIST GAN, the generator receives a structured input (the mask) and must
fill in realistic fluorescence texture while respecting the mask geometry.

Next: evaluate the results quantitatively (`02_Evaluate_pix2pixHD`).